# ROCLING 2026：Batch-controlled ablation（Colab）

這份 notebook 專門補齊論文的 batch-size confounding。四個條件均使用相同的
MacBERT checkpoint、batch size 32、learning rate 2e-5、4 epochs、max length 256、
checkpoint criterion 與三個 seeds（42、1、2）。

| condition | lexicon | augmentation | role |
|---|---|---|---|
| `nolex_b32` | none | none | 純 lexicon ablation 對照 |
| `l1_b32` | L1 | none | lexicon treatment，也是 augmentation baseline |
| `aug_E_b32` | L1 | E graph augmentation | 純 augmentation ablation |
| `aug_F2_b32` | L1 | F2 augmentation | 純 augmentation ablation |

總計 **4 conditions × 3 seeds = 12 runs**。A100 預估約 3 小時，實際時間依 runtime 而異。
每個 run 完成後會立即產生 validation/test submission，將小型結果存入 Google Drive，
並在驗證完成後依設定刪除 runtime 內的大型 checkpoint。

執行方式：依序執行所有 cells。重新連線後重跑前置 cells 與主迴圈，已完成的 run 會由
Drive receipt 自動跳過。

In [1]:
# 1) 安裝套件並確認 GPU
!pip -q install "transformers>=4.40" "huggingface_hub>=0.23" jieba scikit-learn scipy pandas

import platform, subprocess, sys
import torch

assert torch.cuda.is_available(), "沒有 CUDA GPU：請在 Colab 選擇 GPU runtime 後重新執行。"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {GPU_NAME} ({GPU_VRAM_GB:.1f} GB)")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

GPU: NVIDIA A100-SXM4-40GB (39.5 GB)
Sun Aug  2 15:48:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------

In [2]:
# 2) 掛載 Google Drive；所有可交付結果都會持久化到這裡
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
RESULT_ROOT = Path("/content/drive/MyDrive/ROCLING2026_batch_controlled")
RUNS_ROOT = RESULT_ROOT / "runs"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

# 驗證 Drive 寫入權限
probe = RESULT_ROOT / ".write_probe"
probe.write_text("ok", encoding="utf-8")
assert probe.read_text(encoding="utf-8") == "ok"
probe.unlink()
print("結果目錄:", RESULT_ROOT)

Mounted at /content/drive
結果目錄: /content/drive/MyDrive/ROCLING2026_batch_controlled


In [3]:
# 3) 匿名取得公開 repo；首次執行記錄 commit，續跑時固定使用同一 commit
import json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/chen0427ok/DSA-NIFT.git"
REPO_PATH = Path("/content/DSA-NIFT")
ENV_MANIFEST_PATH = RESULT_ROOT / "environment_manifest.json"
old_manifest = json.loads(ENV_MANIFEST_PATH.read_text(encoding="utf-8")) if ENV_MANIFEST_PATH.exists() else {}
pinned_repo_commit = old_manifest.get("repo_commit")

if not (REPO_PATH / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_PATH)], check=True)
actual_remote = subprocess.run(
    ["git", "remote", "get-url", "origin"], cwd=REPO_PATH,
    check=True, capture_output=True, text=True
).stdout.strip()
assert actual_remote.rstrip("/") == REPO_URL.rstrip("/"), f"非預期 remote: {actual_remote}"

if pinned_repo_commit:
    subprocess.run(["git", "fetch", "origin", pinned_repo_commit], cwd=REPO_PATH, check=True)
    subprocess.run(["git", "checkout", "--detach", pinned_repo_commit], cwd=REPO_PATH, check=True)
else:
    subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_PATH, check=True)
    subprocess.run(["git", "checkout", "--detach", "origin/main"], cwd=REPO_PATH, check=True)

REPO_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_PATH,
    check=True, capture_output=True, text=True
).stdout.strip()
if pinned_repo_commit:
    assert REPO_COMMIT == pinned_repo_commit, "repo commit 與既有實驗 manifest 不一致"
os.chdir(REPO_PATH)
print("Repo commit:", REPO_COMMIT)

Repo commit: 4a9eccc2f5ff92b187738b569572049171602063


In [4]:
# 4) 固定同一個 MacBERT revision，並寫入環境 manifest
import importlib.metadata as metadata
from huggingface_hub import model_info, snapshot_download

MODEL_ID = "hfl/chinese-macbert-base"
MODEL_SHA = old_manifest.get("model_sha") or model_info(MODEL_ID).sha
MODEL_DIR = Path("/content/model_snapshot")
snapshot_download(repo_id=MODEL_ID, revision=MODEL_SHA, local_dir=str(MODEL_DIR))

BASE_MANIFEST = {
    "repo_url": REPO_URL,
    "repo_commit": REPO_COMMIT,
    "model_id": MODEL_ID,
    "model_sha": MODEL_SHA,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": metadata.version("transformers"),
    "huggingface_hub": metadata.version("huggingface_hub"),
    "gpu": GPU_NAME,
    "gpu_vram_gb": round(GPU_VRAM_GB, 2),
}
if old_manifest:
    assert old_manifest["repo_commit"] == REPO_COMMIT
    assert old_manifest["model_sha"] == MODEL_SHA
tmp_manifest = ENV_MANIFEST_PATH.with_suffix(".json.tmp")
tmp_manifest.write_text(json.dumps(BASE_MANIFEST, ensure_ascii=False, indent=2), encoding="utf-8")
os.replace(tmp_manifest, ENV_MANIFEST_PATH)
print(json.dumps(BASE_MANIFEST, ensure_ascii=False, indent=2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

{
  "repo_url": "https://github.com/chen0427ok/DSA-NIFT.git",
  "repo_commit": "4a9eccc2f5ff92b187738b569572049171602063",
  "model_id": "hfl/chinese-macbert-base",
  "model_sha": "a986e004d2a7f2a1c2f5a3edef4e20604a974ed1",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "transformers": "5.13.1",
  "huggingface_hub": "1.23.0",
  "gpu": "NVIDIA A100-SXM4-40GB",
  "gpu_vram_gb": 39.49
}


In [5]:
# 5) 唯一的實驗設定矩陣與 preflight
import pandas as pd

SEEDS = [42, 1, 2]
BATCH_SIZE = 32
LR = 2e-5
MAX_LEN = 256
EPOCHS = 4
DELETE_CHECKPOINT_AFTER_PERSIST = True  # True 可節省 runtime 磁碟；權重刪除後只能重訓
STOP_ON_ERROR = True
FORCE_RERUN = False

CONDITIONS = {
    "nolex_b32": {"lex_mode": "none", "extra_train": None},
    "l1_b32": {"lex_mode": "l1", "extra_train": None},
    "aug_E_b32": {"lex_mode": "l1", "extra_train": "data/train_aug_E.csv"},
    "aug_F2_b32": {"lex_mode": "l1", "extra_train": "data/train_aug_F2.csv"},
}

REQUIRED_FILES = [
    "train_v2.py", "predict.py", "lexicon.py",
    "data/train.csv", "data/dev.csv", "data/val_unlabeled.csv",
    "data/train_aug_E.csv", "data/train_aug_F2.csv", "data/DSANIDF_TestSet.csv",
]
missing = [p for p in REQUIRED_FILES if not Path(p).exists()]
assert not missing, f"缺少必要檔案: {missing}"

expected_columns = {
    "data/train.csv": {"id", "text", "valence", "arousal"},
    "data/dev.csv": {"id", "text", "valence", "arousal"},
    "data/val_unlabeled.csv": {"id", "text"},
    "data/train_aug_E.csv": {"id", "text", "valence", "arousal"},
    "data/train_aug_F2.csv": {"id", "text", "valence", "arousal"},
    "data/DSANIDF_TestSet.csv": {"id", "text"},
}
row_counts = {}
for path, needed in expected_columns.items():
    frame = pd.read_csv(path)
    columns = {str(c).strip().lower() for c in frame.columns}
    assert needed <= columns, f"{path} 欄位錯誤: {frame.columns.tolist()}"
    assert len(frame) > 0, f"{path} 是空檔"
    row_counts[path] = len(frame)

matrix = pd.DataFrame([
    {"condition": c, "seed": s, **cfg, "batch_size": BATCH_SIZE,
     "lr": LR, "epochs": EPOCHS, "max_len": MAX_LEN}
    for c, cfg in CONDITIONS.items() for s in SEEDS
])
display(matrix)
print("row counts:", row_counts)
assert len(matrix) == 12

,condition,seed,lex_mode,extra_train,batch_size,lr,epochs,max_len
0,nolex_b32,42,none,None,32,0.00002,4,256
1,nolex_b32,1,none,None,32,0.00002,4,256
2,nolex_b32,2,none,None,32,0.00002,4,256
3,l1_b32,42,l1,None,32,0.00002,4,256
4,l1_b32,1,l1,None,32,0.00002,4,256
5,l1_b32,2,l1,None,32,0.00002,4,256
6,aug_E_b32,42,l1,data/train_aug_E.csv,32,0.00002,4,256
7,aug_E_b32,1,l1,data/train_aug_E.csv,32,0.00002,4,256
8,aug_E_b32,2,l1,data/train_aug_E.csv,32,0.00002,4,256
9,aug_F2_b32,42,l1,data/train_aug_F2.csv,32,0.00002,4,256


row counts: {'data/train.csv': 9435, 'data/dev.csv': 253, 'data/val_unlabeled.csv': 200, 'data/train_aug_E.csv': 400, 'data/train_aug_F2.csv': 400, 'data/DSANIDF_TestSet.csv': 1100}


In [6]:
# 6) 訓練、推論、Drive receipt 與結果打包 helpers
import csv, hashlib, json, os, shutil, subprocess, sys, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path

def run_name(condition, seed):
    return f"controlled_{condition}_s{seed}"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def atomic_json(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, path)

def execute_logged(command, log_path):
    print("$", " ".join(map(str, command)), flush=True)
    with open(log_path, "w", encoding="utf-8") as log:
        process = subprocess.Popen(
            list(map(str, command)), cwd=REPO_PATH,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        rc = process.wait()
    if rc != 0:
        raise RuntimeError(f"command failed (rc={rc}): {' '.join(map(str, command))}")

def build_train_command(condition, seed):
    cfg = CONDITIONS[condition]
    cmd = [
        sys.executable, "train_v2.py",
        "--model", str(MODEL_DIR),
        "--run_name", run_name(condition, seed),
        "--lex_mode", cfg["lex_mode"],
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--lr", str(LR),
        "--max_len", str(MAX_LEN),
    ]
    if cfg["extra_train"]:
        cmd += ["--extra_train", cfg["extra_train"]]
    return cmd

def dev_metrics(path):
    df = pd.read_csv(path)
    result = {}
    for dim, prefix in [("valence", "V"), ("arousal", "A")]:
        gold = df[f"{dim}_true"].to_numpy(float)
        pred = df[f"{dim}_pred"].to_numpy(float)
        result[f"{prefix}_MAE"] = float(np.mean(np.abs(pred - gold)))
        result[f"{prefix}_PCC"] = float(np.corrcoef(pred, gold)[0, 1])
    return result

def zip_submission(csv_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname="submission.csv")
    with zipfile.ZipFile(zip_path) as zf:
        assert zf.namelist() == ["submission.csv"]

def valid_receipt(run):
    run_dir = RUNS_ROOT / run
    receipt_path = run_dir / "receipt.json"
    if not receipt_path.exists():
        return False
    try:
        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
        for name, expected_hash in receipt["artifact_sha256"].items():
            path = run_dir / name
            if not path.exists() or path.stat().st_size == 0 or sha256(path) != expected_hash:
                return False
        return receipt["status"] == "complete"
    except Exception as exc:
        print(f"receipt invalid for {run}: {exc}")
        return False

def run_one(condition, seed):
    run = run_name(condition, seed)
    if valid_receipt(run) and not FORCE_RERUN:
        print(f"⏭ {run}: Drive receipt 已完成")
        return "skipped"

    cfg = CONDITIONS[condition]
    run_dir = RUNS_ROOT / run
    run_dir.mkdir(parents=True, exist_ok=True)
    local_log = Path("/content") / f"{run}.log"
    started = time.time()
    train_cmd = build_train_command(condition, seed)
    execute_logged(train_cmd, local_log)

    ckpt = REPO_PATH / "outputs" / f"{run}_best.pt"
    dev_pred = REPO_PATH / "outputs" / "preds" / f"{run}_dev.csv"
    val_pred = REPO_PATH / "outputs" / "preds" / f"{run}_val.csv"
    val_sub = REPO_PATH / "outputs" / f"{run}_submission.csv"
    for path in [ckpt, dev_pred, val_pred, val_sub]:
        assert path.exists() and path.stat().st_size > 0, f"缺少訓練產物: {path}"

    predict_cmd = [
        sys.executable, "predict.py", "--ckpt", str(ckpt),
        "--model", str(MODEL_DIR), "--lex_mode", cfg["lex_mode"],
        "--input", "data/DSANIDF_TestSet.csv", "--run_name", run,
        "--split", "test", "--batch_size", str(BATCH_SIZE),
        "--max_len", str(MAX_LEN),
    ]
    execute_logged(predict_cmd, local_log.with_name(f"{run}_predict.log"))
    test_pred = REPO_PATH / "outputs" / "preds" / f"{run}_test.csv"
    test_sub = REPO_PATH / "outputs" / f"{run}_test_submission.csv"
    for path in [test_pred, test_sub]:
        assert path.exists() and path.stat().st_size > 0, f"缺少 test 產物: {path}"

    val_zip = Path("/content") / f"{run}_validation_submission.zip"
    test_zip = Path("/content") / f"{run}_test_submission.zip"
    zip_submission(val_sub, val_zip)
    zip_submission(test_sub, test_zip)
    metrics = dev_metrics(dev_pred)
    config = {
        **BASE_MANIFEST, "run_name": run, "condition": condition, "seed": seed,
        "lex_mode": cfg["lex_mode"], "extra_train": cfg["extra_train"],
        "batch_size": BATCH_SIZE, "lr": LR, "epochs": EPOCHS,
        "max_len": MAX_LEN, "train_command": train_cmd,
        "predict_command": predict_cmd, "elapsed_minutes": (time.time() - started) / 60,
    }
    local_config = Path("/content") / f"{run}_config.json"
    local_metrics = Path("/content") / f"{run}_dev_metrics.json"
    local_config.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    local_metrics.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

    artifacts = {
        "train.log": local_log,
        "predict.log": local_log.with_name(f"{run}_predict.log"),
        "config.json": local_config,
        "dev_metrics.json": local_metrics,
        "dev_predictions.csv": dev_pred,
        "validation_predictions.csv": val_pred,
        "test_predictions.csv": test_pred,
        "validation_submission.csv": val_sub,
        "test_submission.csv": test_sub,
        "validation_submission.zip": val_zip,
        "test_submission.zip": test_zip,
    }
    for name, source in artifacts.items():
        shutil.copy2(source, run_dir / name)
    artifact_hashes = {name: sha256(run_dir / name) for name in artifacts}
    receipt = {
        "status": "complete", "run_name": run, "condition": condition, "seed": seed,
        "dev_metrics": metrics, "artifact_sha256": artifact_hashes,
        "repo_commit": REPO_COMMIT, "model_sha": MODEL_SHA,
    }
    atomic_json(run_dir / "receipt.json", receipt)
    assert valid_receipt(run), f"Drive persistence verification failed: {run}"

    if DELETE_CHECKPOINT_AFTER_PERSIST:
        ckpt.unlink()
        print(f"🧹 已驗證 Drive artifacts，刪除 runtime checkpoint: {ckpt.name}")
    print(f"✅ {run}: {metrics}")
    return "completed"

In [ ]:
# 7) 主迴圈：12 runs。中斷後重跑本 cell 即可續跑。
statuses = []
for condition in CONDITIONS:
    for seed in SEEDS:
        run = run_name(condition, seed)
        print("\n" + "=" * 80)
        print("▶", run)
        print("=" * 80)
        try:
            status = run_one(condition, seed)
            statuses.append({"run_name": run, "status": status})
        except Exception as exc:
            statuses.append({"run_name": run, "status": "failed", "error": repr(exc)})
            print(f"❌ {run}: {exc}")
            if STOP_ON_ERROR:
                raise
display(pd.DataFrame(statuses))


▶ controlled_nolex_b32_s42
$ /usr/bin/python3 train_v2.py --model /content/model_snapshot --run_name controlled_nolex_b32_s42 --lex_mode none --seed 42 --epochs 4 --batch_size 32 --lr 2e-05 --max_len 256
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
run=controlled_nolex_b32_s42 | device=cuda | model=/content/model_snapshot | lex=none | aw=1.0 pcc_w=0.0 | src_aware=False | rank_aug=None | pooling=mean
train=9435 dev=253 val=200
lexicon dim = 0

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17747.53it/s]
[transformers] BertMo

,run_name,status
0,controlled_nolex_b32_s42,completed
1,controlled_nolex_b32_s1,completed
2,controlled_nolex_b32_s2,completed
3,controlled_l1_b32_s42,completed
4,controlled_l1_b32_s1,completed
5,controlled_l1_b32_s2,completed
6,controlled_aug_E_b32_s42,completed
7,controlled_aug_E_b32_s1,completed
8,controlled_aug_E_b32_s2,completed
9,controlled_aug_F2_b32_s42,completed


In [ ]:
# 8) 嚴格彙整、建立官方分數填寫表與 handoff zip
import zipfile

records = []
for condition in CONDITIONS:
    for seed in SEEDS:
        run = run_name(condition, seed)
        if not valid_receipt(run):
            continue
        receipt = json.loads((RUNS_ROOT / run / "receipt.json").read_text(encoding="utf-8"))
        records.append({
            "run_name": run, "condition": condition, "seed": seed,
            **receipt["dev_metrics"], "repo_commit": receipt["repo_commit"],
            "model_sha": receipt["model_sha"],
        })

per_seed = pd.DataFrame(records).sort_values(["condition", "seed"]) if records else pd.DataFrame()
per_seed_path = RESULT_ROOT / "dev_metrics_per_seed.csv"
per_seed.to_csv(per_seed_path, index=False)

summary_rows = []
for condition in CONDITIONS:
    part = per_seed[per_seed["condition"] == condition] if not per_seed.empty else pd.DataFrame()
    observed = set(part["seed"].astype(int)) if not part.empty else set()
    complete = observed == set(SEEDS)
    row = {"condition": condition, "n_seed": len(observed), "status": "complete" if complete else "incomplete"}
    for metric in ["V_MAE", "V_PCC", "A_MAE", "A_PCC"]:
        row[f"{metric}_mean"] = part[metric].mean() if complete else np.nan
        row[f"{metric}_sd"] = part[metric].std(ddof=1) if complete else np.nan
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary_path = RESULT_ROOT / "dev_metrics_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)

score_rows = [
    {"run_name": run_name(c, s), "condition": c, "seed": s,
     "V_MAE": "", "V_PCC": "", "A_MAE": "", "A_PCC": ""}
    for c in CONDITIONS for s in SEEDS
]
scores_path = RESULT_ROOT / "official_scores_to_fill.csv"
pd.DataFrame(score_rows).to_csv(scores_path, index=False)
matrix.to_csv(RESULT_ROOT / "run_manifest.csv", index=False)

incomplete = summary[summary["status"] != "complete"]["condition"].tolist()
if incomplete:
    print("⚠️ 尚未完成，不計算缺 seed 條件的 mean±SD:", incomplete)
    print("請重跑主迴圈 cell；handoff zip 仍會建立，方便診斷。")

local_handoff = Path("/content/controlled_ablation_handoff.zip")
if local_handoff.exists():
    local_handoff.unlink()
with zipfile.ZipFile(local_handoff, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in RESULT_ROOT.rglob("*"):
        if path.is_file() and path.name != "controlled_ablation_handoff.zip":
            zf.write(path, arcname=str(path.relative_to(RESULT_ROOT)))
drive_handoff = RESULT_ROOT / "controlled_ablation_handoff.zip"
shutil.copy2(local_handoff, drive_handoff)
print("\nHandoff:", drive_handoff)
print("Official score sheet:", scores_path)
print(f"完成 receipts: {len(records)}/12")

,condition,n_seed,status,V_MAE_mean,V_MAE_sd,V_PCC_mean,V_PCC_sd,A_MAE_mean,A_MAE_sd,A_PCC_mean,A_PCC_sd
0,nolex_b32,3,complete,0.474449,0.003337,0.813113,0.005708,0.840964,0.006779,0.601424,0.002839
1,l1_b32,3,complete,0.484196,0.007291,0.813350,0.002147,0.851872,0.011604,0.595316,0.009243
2,aug_E_b32,3,complete,0.488111,0.013298,0.809534,0.006788,0.847416,0.016166,0.599397,0.012547
3,aug_F2_b32,3,complete,0.487867,0.015977,0.809650,0.011491,0.850358,0.003525,0.601943,0.005720



Handoff: /content/drive/MyDrive/ROCLING2026_batch_controlled/controlled_ablation_handoff.zip
Official score sheet: /content/drive/MyDrive/ROCLING2026_batch_controlled/official_scores_to_fill.csv
完成 receipts: 12/12


## 訓練完成後要給我的資料

1. 將 12 份 validation/test submission zip 上傳官方 leaderboard，取得每個 run 的
   V-MAE、V-PCC、A-MAE、A-PCC。
2. 把分數填入 Google Drive 中的 `official_scores_to_fill.csv`。
3. 將填好的 CSV 或整包 `controlled_ablation_handoff.zip` 給我。

收到官方數據後，我會比較：

- `nolex_b32` vs `l1_b32`：純 lexicon effect；
- `l1_b32` vs `aug_E_b32` vs `aug_F2_b32`：純 augmentation effect；
- 各條件的 3-seed mean、sample SD 與效應方向。

之後再依控制實驗證據更新 `paper/main.tex`、表格與限制段落，刪除或改寫
`batch-confounded` 防禦性文字。